In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format='retina'
import daq
import matplotlib.pyplot as plt

In [ ]:
import spectrum as sp

daq.list_acqs()                                # one row per acq: energies, times, sample, comments
s = daq.load_acq(38, calibrated=True)          # DAQ's dark-corrected spectrum, in eV
s = s.lowpass().smooth_gauss(0.05).normalised()          # every operation returns a new Spectrum

In [ ]:
import lmfit
import fitting_functions as ff
w = s.cropped(522, 527)                               # fit window, in eV
params = ff.make_parameters([524.0, 525.5],           # initial positions
                            amp=(0.1, 0, 1),         # (value, min, max), in units of the data
                            width=(0.5, 0.01, 1),     # Gaussian FWHM, eV
                            position_window=2)      # each position may move +-1 eV
result = lmfit.minimize(ff.residual, params, args=(w.energy, w.intensity, 2, "gauss"))
lmfit.report_fit(result)

In [ ]:
get_gauss = lambda e, res, idx: ff.gauss(e, res.params[f'position_{idx}'], res.params[f'amp_{idx}'], res.params[f'width_{idx}'])

In [ ]:
len(result.params)//3

In [ ]:
s.plot()
for idx in range(len(result.params)//3):
    plt.plot(s.energy, get_gauss(s.energy, result, idx))
plt.xlabel("Energy (eV)")
plt.ylabel("Intensity (a.u.)")
plt.show()